# Week 1 · Notebook 00: Python for Astronomers, a Crash Course
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nightwings24/exoplanet-transit-winter-project/blob/main/notebooks/00_python_crash_course.ipynb)

**Who this is for:** anyone who has never written Python, or wrote some once and forgot.
Seniors: skim it, do the 🔴 exercises, then help someone next to you.
**Time:** 3–4 hours, at your own pace.

### 🎯 By the end of this notebook you can
1. run a notebook and read an error message without panicking;
2. store numbers, text, lists and dictionaries in variables;
3. write a function and a loop;
4. do maths on thousands of numbers at once with **NumPy**;
5. pick out data with a **boolean mask** (you will use this every week);
6. explain why averaging *N* noisy points shrinks the noise by $\sqrt{N}$;
7. make a properly labelled plot;
8. build a fake planet transit from scratch and decide whether it is detectable.

### 🧭 Roadmap
| § | Topic | Why it matters for finding planets |
|---|---|---|
| 0 | How notebooks work | You will live in notebooks for four weeks |
| 1 | Variables and numbers | Transit depth is one line of arithmetic |
| 2 | Reading error messages | You *will* see errors. Reading them is a skill |
| 3 | Lists, loops, `if`, dictionaries | Repeating a calculation for many planets |
| 4 | Functions | Write the physics once, reuse it everywhere |
| 5 | NumPy arrays | A light curve is an array of ~10,000 numbers |
| 6 | Boolean masks | Selecting in-transit points, removing outliers |
| 7 | Noise and averaging | The single idea behind every detection |
| 8 | Plotting | Every result in your report is a figure |
| 9 | Mini-project: a fake transit | Everything above, combined |
<details><summary><b>📎 How to read this notebook (click to open)</b></summary>

| Symbol | Meaning |
|---|---|
| 🔮 **Predict** | Before running the next cell, write down what you expect. Being wrong is how you learn. |
| 👀 **What to look for** | How to read the output or plot you just made. |
| ✏️ **Your turn** | An exercise. 🟢 everyone · 🟡 stretch · 🔴 challenge. |
| 💡 / ✅ | Click to reveal a hint or a worked solution. Try first, then peek. |
| ⚠️ **Pitfall** | A mistake that real researchers make. |
| 🧠 **Concept check** | Short quiz. Answers are hidden under each question. |
| `check(...)` | Self-check: prints ✅ when your exercise answer is right. |

Run cells **top to bottom** with **Shift+Enter**. If things get weird: *Runtime → Restart and run all* (Colab) or *Kernel → Restart & Run All* (Jupyter).
</details>

In [ ]:
# ⚙️ SETUP: run this cell first, every time you open the notebook. Works in Google Colab and locally.
import os, sys
REPO_URL = "https://github.com/Nightwings24/exoplanet-transit-winter-project.git"  # <- change to YOUR repo
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("exoplanet-transit-winter-project"):
        !git clone -q {REPO_URL}
    %cd exoplanet-transit-winter-project
    !pip install -q lightkurve batman-package wotan tqdm
# Make `import transitpipe` work without installing the package
ROOT = os.path.abspath("." if os.path.exists("src") else "..")
sys.path.insert(0, os.path.join(ROOT, "src"))
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (10, 3.5)})

def check(name, value, expected, rtol=0.02):
    """Self-check for the exercises. Prints ✅, ❌, or ⏳ (not attempted yet)."""
    if value is None:
        print(f"⏳ {name}: not attempted yet (replace None with your answer)")
        return
    try:
        ok = bool(np.allclose(value, expected, rtol=rtol))
    except Exception:
        ok = value == expected
    print(f"✅ {name}: correct!" if ok else f"❌ {name}: not quite (you gave {value!r}). Check the hint.")

print("ready, repo root:", ROOT)

## 0. How notebooks work
A notebook is a list of **cells**. *Markdown* cells (like this one) hold text. *Code* cells hold Python.
Run a code cell with **Shift+Enter**. Output appears under it.

Two rules that explain 90 % of beginner confusion:
1. **The last line of a cell is displayed automatically.** Anything else needs `print(...)`.
2. **Order matters, not position.** Python remembers what you *ran*, in the order you ran it. If you
   skip a cell, or run cells out of order, variables may be missing or have old values.

Anything after a `#` is a **comment**: Python ignores it. Comments are notes for humans.

In [ ]:
print("Hello, astronomers!")   # print() shows something on the screen
2 + 3                          # last line: shown automatically, no print needed

In [ ]:
x = 5          # store the number 5 under the name x (nothing is shown: an assignment is not a value)
x * 2          # x is remembered from the line above

🔮 **Predict:** if you now change `x = 5` to `x = 50` in the cell above but run *only the cell below*,
what will it print? Try it. Then run the cell above, and the one below again.

In [ ]:
print("x is", x)

## 1. Variables, numbers and units
A **variable** is a name for a value. Python has a few basic types:

| type | example | used for |
|---|---|---|
| `int` | `6371` | whole numbers |
| `float` | `0.00916`, `1e-6` | real numbers. `1e-6` means $1\times10^{-6}$ |
| `str` | `"Kepler-10"` | text (always in quotes) |
| `bool` | `True`, `False` | yes/no answers |

Python does not know about units. **You** must keep track of them, and the safest way is to put the
unit in the variable name (`R_sun_km`, `period_days`).

In [ ]:
R_sun_km = 695_700          # radius of the Sun in km (underscores are just for readability)
R_earth_km = 6_371
R_jupiter_km = 69_911
star_name = "Sun"
has_planets = True

print(type(R_sun_km), type(0.5), type(star_name), type(has_planets))

ratio = R_earth_km / R_sun_km
print("Earth/Sun radius ratio:", ratio)
print(f"Same thing, formatted: {ratio:.5f}   in scientific notation: {ratio:.2e}")

The `f"...{ratio:.5f}..."` is an **f-string**: Python replaces `{...}` with the value. `:.5f` means
"5 decimals", `:.2e` means "scientific notation, 2 decimals". You will use f-strings constantly.

### 🔭 Physics moment: the transit depth
When a planet passes in front of its star it blocks a small disk of the star's light. The fractional dip in
brightness, the **transit depth** $\delta$, is the ratio of the two disk areas:

$$\delta = \frac{\pi R_p^2}{\pi R_\star^2} = \left(\frac{R_p}{R_\star}\right)^2$$

Areas go as radius *squared*, so small planets make *very* small dips. Astronomers quote tiny depths in
**ppm** (parts per million): multiply the fraction by $10^6$. 1 % = 10,000 ppm.

In [ ]:
depth_earth = (R_earth_km / R_sun_km) ** 2      # ** means "to the power of"
depth_jup = (R_jupiter_km / R_sun_km) ** 2
print(f"Earth transiting the Sun:   {depth_earth * 1e6:.0f} ppm")
print(f"Jupiter transiting the Sun: {depth_jup * 100:.2f} %")

👀 **What to look for:** Earth makes an ~84 ppm dip; Jupiter about 1 %. That factor of ~120 is why the
first exoplanets found by transits were hot Jupiters, and why finding Earths needed a space telescope.

### ✏️ Your turn 1 🟢
A "super-Earth" with radius 1.6 R⊕ orbits a small M-dwarf star with radius 0.3 R☉.
Compute its transit depth **in ppm** and store it in `depth_superearth_ppm`.

<details><summary>💡 Hint</summary>

First compute both radii in km (`1.6 * R_earth_km` and `0.3 * R_sun_km`), then use the depth formula,
then multiply by `1e6`.
</details>

In [ ]:
# ✏️ YOUR CODE HERE: replace None
depth_superearth_ppm = None

check("Your turn 1", depth_superearth_ppm, (1.6 * 6371 / (0.3 * 695700)) ** 2 * 1e6)

<details><summary>✅ Solution and discussion</summary>

```python
R_planet_km = 1.6 * R_earth_km
R_star_km = 0.3 * R_sun_km
depth_superearth_ppm = (R_planet_km / R_star_km) ** 2 * 1e6   # ≈ 2385 ppm
```
The same planet around the Sun would give ≈ 215 ppm. Around a star 3× smaller, the depth is 3² = 9× bigger.
That is why M dwarfs are favourite targets for finding small planets (TRAPPIST-1, TOI-700, LHS 1140…).
</details>

## 2. Reading error messages (a survival skill)
Errors are normal. Professionals see dozens a day. The trick is to **read the last line first**: it names
the error type and usually says exactly what is wrong. The lines above it (the *traceback*) show *where*.

The cell below makes an error on purpose, and catches it so the notebook keeps running.

In [ ]:
try:
    print(R_mars_km)          # we never defined R_mars_km!
except NameError as err:
    print("Python says ->", type(err).__name__ + ":", err)

| Last line says | What it usually means | Typical fix |
|---|---|---|
| `NameError: name 'x' is not defined` | You never ran the cell that creates `x`, or misspelt it | Run cells from the top; check spelling (Python is case-sensitive) |
| `SyntaxError` | Python can't parse the line: missing `)`, `:` or quote | Look at the line *and the one above* |
| `IndentationError` | Spaces at the start of a line are wrong | Use 4 spaces inside `for`/`if`/`def` blocks |
| `TypeError: unsupported operand` | Mixing types, e.g. `"3" + 4` | Convert: `float("3") + 4` |
| `IndexError: index 10 is out of bounds` | Asked for item 10 of a shorter list | Check `len(...)`; remember counting starts at 0 |
| `ModuleNotFoundError: No module named 'x'` | Package not installed / setup cell not run | Run the setup cell; `pip install x` |
| `KeyError: 'rstar'` | Dictionary has no such key | Print `d.keys()` |

⚠️ **Pitfall:** pasting an error into a search engine or an AI assistant is fine, but **read it yourself
first**. You will learn far faster, and you will be able to tell when the AI's fix is wrong.

## 3. Lists, loops, `if`, and dictionaries
A **list** holds several values in order. Python counts from **0**: the first item is `[0]`, the last is `[-1]`.
A **slice** `[a:b]` gives items `a, a+1, …, b-1` (the stop is *excluded*).

In [ ]:
planets = ["Mercury", "Venus", "Earth", "Mars"]
radii = [0.383, 0.949, 1.000, 0.532]          # in Earth radii, same order as `planets`

print(planets[0], "and", planets[-1])         # first and last
print(planets[1:3])                           # items 1 and 2
print("number of planets:", len(planets))

A **`for` loop** repeats indented code once per item. `zip` walks through two lists side by side.
An **`if / elif / else`** chooses what to do. Note the colons `:` and the 4-space indentation: they are
how Python knows what is inside the loop.

In [ ]:
for name, r in zip(planets, radii):
    if r < 0.6:
        size = "small"
    elif r < 1.5:
        size = "Earth-like"
    else:
        size = "big"
    print(f"{name:8s} {r:5.3f} R_earth -> {size}")

A **dictionary** stores values under names (*keys*). It's the natural way to keep a star's properties
together. You will use one called `STAR` in every later notebook.

In [ ]:
star = {"name": "Kepler-10", "rstar": 1.065, "mstar": 0.910}   # radius in R_sun, mass in M_sun
print(star["name"], "has radius", star["rstar"], "R_sun")
star["teff"] = 5708            # add a new key: effective temperature in K
print(star.keys())

## 4. Functions: write the physics once
A **function** packages a calculation. `def` starts it, the indented block is the body, `return` hands back
the answer. The text in triple quotes is a *docstring*: it documents what the function does and its units.

In [ ]:
def transit_depth(rp_rearth, rstar_rsun):
    """Fractional transit depth for a planet (Earth radii) and star (solar radii)."""
    rp_km = rp_rearth * R_earth_km
    rs_km = rstar_rsun * R_sun_km
    return (rp_km / rs_km) ** 2

for name, rp in [("Mars-size", 0.53), ("Earth", 1.0), ("Neptune", 3.9), ("Jupiter", 11.2)]:
    print(f"{name:10s} {transit_depth(rp, 1.0) * 1e6:9.0f} ppm")

### ✏️ Your turn 2 🟢
In real life we go the *other* way: we **measure** the depth and want the planet's radius.
Complete `radius_from_depth` so it returns the planet radius in Earth radii.

<details><summary>💡 Hint</summary>

Invert $\delta = (R_p/R_\star)^2$: $R_p = \sqrt{\delta}\,R_\star$. Square root: `depth ** 0.5` or `np.sqrt(depth)`.
Then convert from km to Earth radii by dividing by `R_earth_km`.
</details>

In [ ]:
def radius_from_depth(depth, rstar_rsun):
    """Planet radius in Earth radii from a fractional transit depth and a star radius in R_sun."""
    # ✏️ YOUR CODE HERE: replace None with the formula
    return None

check("Your turn 2", radius_from_depth(transit_depth(2.0, 1.0), 1.0), 2.0)

<details><summary>✅ Solution</summary>

```python
def radius_from_depth(depth, rstar_rsun):
    return depth ** 0.5 * rstar_rsun * R_sun_km / R_earth_km
```
The self-check *round-trips*: depth of a 2 R⊕ planet → back to radius → must give 2. Round-trip tests
are a great habit. The package has the same function, `transitpipe.utils.depth_to_rearth`.
</details>

## 5. NumPy: maths on whole arrays at once
A light curve is ~10,000 numbers. **NumPy arrays** let you do maths on all of them in one line, fast, with
no loop. This is the single most important tool in the whole project.

In [ ]:
time = np.linspace(0, 10, 11)         # 11 evenly spaced numbers from 0 to 10
print(time)
print(time * 2)                       # every element doubled: no loop needed
print(np.sin(time).round(2))          # every element through sin()
print("mean:", time.mean(), " max:", time.max(), " length:", len(time))
print("first three:", time[:3], " last:", time[-1])

### Mean versus median: why we love the median
🔮 **Predict:** the array below is 7 normal measurements and one bad one (a cosmic-ray hit).
Which will the bad point move more, the mean or the median?

In [ ]:
flux = np.array([1.001, 0.999, 1.000, 1.002, 0.998, 1.000, 1.001, 1.350])   # last point is junk
print(f"mean   = {flux.mean():.4f}")
print(f"median = {np.median(flux):.4f}")

👀 The mean jumped by 4 %; the median barely moved. The median is **robust** to outliers. Next week's
detrending uses a *sliding median* for exactly this reason: a transit is a handful of "outlier" points
that we do **not** want the trend to follow.

### NaN: "not a number"
Real data have missing values, stored as `np.nan`. Any maths with a NaN gives NaN. Use the `nan`-aware
versions (`np.nanmean`, `np.nanmedian`) or remove them with a mask (next section).

In [ ]:
a = np.array([1.0, np.nan, 3.0])
print(a.mean(), np.nanmean(a), np.isnan(a))

### ✏️ Your turn 3 🟢
`t` holds a time every half day from 0 up to (not including) 10. Fill in the three answers using
NumPy, not by counting.

<details><summary>💡 Hint</summary>

`len(t)`, `t[4]`, and `t[-3:]` (a slice from the third-last item to the end).
</details>

In [ ]:
t = np.arange(0, 10, 0.5)       # arange(start, stop, step): stop is excluded

n_points = None        # how many elements?
fifth_value = None     # the 5th element (careful: counting starts at 0!)
last_three = None      # the last three elements, as an array

check("number of points", n_points, 20)
check("fifth value", fifth_value, 2.0)
check("last three", last_three, [8.5, 9.0, 9.5])

## 6. Boolean masks: the trick you will use every single week
Compare an array with a number and you get an array of `True`/`False`, called a **mask**.
Use a mask inside `[ ]` to **select** the elements where it is `True`.

| code | meaning |
|---|---|
| `flux < 0.97` | True where flux is below 0.97 |
| `mask.sum()` | how many are True (True counts as 1) |
| `flux[mask]` | only the selected values |
| `~mask` | NOT: flips True and False |
| `(a > 1) & (a < 3)` | AND (brackets are required!) |
| `(a < 1) \| (a > 3)` | OR |

In [ ]:
time = np.arange(8.0)                                      # 0, 1, ..., 7 days
flux = np.array([1.00, 1.01, 0.99, 0.95, 0.94, 0.95, 1.00, 1.02])
low = flux < 0.97
print("mask:          ", low)
print("low fluxes:    ", flux[low])
print("WHEN they were:", time[low])           # a mask from one array can select from another!
print("how many:      ", low.sum())
print("everything else:", flux[~low])

👀 The line `time[low]` is the key idea: build the mask from **flux**, apply it to **time**, and you
get the times of the dip. Next week you'll build masks like *"within 1 hour of a predicted transit"* and use
them to hide transits, measure depths, and remove junk.

### ✏️ Your turn 4 🟡
Using the `time` and `flux` arrays above, count the points that are **between day 2 and day 5
(inclusive)** *and* have flux **below 0.96**.

<details><summary>💡 Hint</summary>

Three conditions joined with `&`, each in brackets: `(time >= 2) & (time <= 5) & (flux < 0.96)`, then `.sum()`.
</details>

In [ ]:
n_selected = None    # ✏️ YOUR CODE HERE

check("Your turn 4", n_selected, 3)

## 7. Noise, and why averaging finds planets
Every measurement has random **noise**. NumPy can make fake noise with a *random number generator*.
`rng.normal(0, sigma, N)` draws `N` numbers from a bell curve (Gaussian) of width `sigma`.

Here is the most important idea of the whole project: if each point has noise $\sigma$, the **average of
$N$ points** has noise $\sigma/\sqrt{N}$. Let's check it by experiment instead of believing it.

In [ ]:
rng = np.random.default_rng(1)       # a seeded generator: same "random" numbers every run
sigma = 400e-6                        # 400 ppm noise per measurement
Ns = [1, 4, 16, 64, 256, 1024]
measured = []
for N in Ns:
    # 2000 experiments, each averaging N noisy points; how much do those averages scatter?
    averages = rng.normal(0, sigma, size=(2000, N)).mean(axis=1)
    measured.append(averages.std())
    print(f"N = {N:5d}: scatter of the average = {averages.std() * 1e6:6.1f} ppm   (σ/√N = {sigma / np.sqrt(N) * 1e6:6.1f})")

plt.figure(figsize=(5, 3.5))
plt.loglog(Ns, np.array(measured) * 1e6, "o", label="experiment")
plt.loglog(Ns, sigma / np.sqrt(Ns) * 1e6, "-", label=r"$\sigma/\sqrt{N}$")
plt.xlabel("number of points averaged, N"); plt.ylabel("noise of the average [ppm]")
plt.legend(); plt.show()

👀 Each point is noisy at 400 ppm, but the average of 1024 points is good to ~12 ppm.
So a planet whose dip is *smaller than the noise on every single point* can still be detected, as long as we
can average enough in-transit points. That leads to the **signal-to-noise ratio** of a transit:

$$\text{SNR} \approx \frac{\delta}{\sigma}\sqrt{N_\text{in-transit}}$$

Surveys typically need SNR ≳ 7 to believe a detection. Keep this formula in mind all month.

## 8. Plotting with Matplotlib
Club rule: **every plot gets axis labels with units.** A plot without labels is a doodle.
A good figure also has: a title or caption that says what it shows, readable font, and a legend if there is
more than one thing on it.

In [ ]:
t = np.linspace(0, 10, 500)
f = 1 + 0.01 * np.sin(2 * np.pi * t / 3.0)
f_noisy = f + rng.normal(0, 0.003, t.size)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))          # 1 row, 2 panels
ax[0].plot(t, f_noisy, ".", ms=2, color="0.5", label="measurements")
ax[0].plot(t, f, "-", color="C3", label="true signal")
ax[0].set_xlabel("Time [days]"); ax[0].set_ylabel("Normalised flux")
ax[0].set_title("A star that varies with a 3-day period"); ax[0].legend()

ax[1].hist((f_noisy - f) * 1e3, bins=30)
ax[1].set_xlabel("measurement − truth [ppt]"); ax[1].set_ylabel("count")
ax[1].set_title("The noise is Gaussian")
plt.tight_layout()
fig.savefig(os.path.join(ROOT, "figures", "00_example.png"), dpi=150)   # save for your report
plt.show()

Line styles: `"."` dots, `"-"` line, `"o"` circles. Colours: `"C0"`, `"C1"`… are the default cycle,
`"0.5"` is grey. `ms` = marker size. `fig.savefig(...)` writes a PNG you can put in your report.

## 9. Mini-project: build a fake transit light curve from scratch
Real data come next notebook. First, make data where **you know the answer**. This idea, *testing your
method on fake data with a known answer*, is the whole point of Week 4.

The trickiest line is the one that finds "how far is each time from the nearest transit". Let's unpack it on
three numbers first. With period $P = 3.5$ d and first transit at $t_0 = 1.2$ d, transits happen at
1.2, 4.7, 8.2, … d.

In [ ]:
P, t0 = 3.5, 1.2
t_demo = np.array([1.2, 4.6, 6.0, 8.2])
dt = ((t_demo - t0 + 0.5 * P) % P) - 0.5 * P      # % is the remainder ("modulo")
for ti, d in zip(t_demo, dt):
    print(f"t = {ti:4.1f} d  ->  {d:+.2f} d from the nearest transit")

👀 `t = 4.6` is 0.1 d *before* the second transit (−0.10); `t = 6.0` is 1.3 d after the second (+1.30).
Shifting by half a period before `%` and back after makes the answer run from −P/2 to +P/2, centred on
a transit. This is **phase folding**, and it's in every notebook from now on (`transitpipe.utils.phase_fold`).

In [ ]:
rng = np.random.default_rng(42)

cadence_min = 30                              # one measurement every 30 minutes
time = np.arange(0, 27, cadence_min / 1440)   # 27 days (one TESS sector); 1440 minutes per day
period, t0, duration = 3.5, 1.2, 0.12         # days
depth = transit_depth(2.5, 1.0)               # a 2.5 R_earth planet around a Sun-like star
noise_ppm = 400

dt = np.abs(((time - t0 + 0.5 * period) % period) - 0.5 * period)   # distance to nearest transit
in_transit = dt < duration / 2                                       # a boolean mask!

flux = np.ones_like(time)
flux[in_transit] -= depth                                  # the box-shaped dip
flux += rng.normal(0, noise_ppm * 1e-6, len(time))         # add Gaussian noise

plt.plot(time, flux, ".", ms=2, color="k")
plt.xlabel("Time [days]"); plt.ylabel("Normalised flux")
plt.title(f"Can you see the {depth * 1e6:.0f} ppm transits? (noise = {noise_ppm} ppm per point)")
plt.show()

Probably not by eye: each point's noise (400 ppm) is about as big as the dip (~530 ppm).
**Folding** the light curve on the right period stacks all the transits on top of each other, so we can
average them:

In [ ]:
phase_days = ((time - t0 + 0.5 * period) % period) - 0.5 * period
plt.plot(phase_days * 24, (flux - 1) * 1e6, ".", ms=3, alpha=0.4, label="data")

# average the points in 30-minute bins so the dip stands out
bins = np.arange(-12, 12.5, 0.5)
idx = np.digitize(phase_days * 24, bins)
binned = [((flux[idx == i] - 1) * 1e6).mean() for i in range(1, len(bins))]
plt.plot(bins[:-1] + 0.25, binned, "o-", color="C3", label="30-min averages")
plt.xlabel("Hours from mid-transit"); plt.ylabel("Flux − 1 [ppm]")
plt.xlim(-12, 12); plt.title("Folded on the true period"); plt.legend(); plt.show()

N = in_transit.sum()
snr = depth / (noise_ppm * 1e-6) * np.sqrt(N)
print(f"{N} in-transit points -> expected SNR ≈ {snr:.1f}")

### ✏️ Your turn 5 (the important one)
1. 🟢 Fold on a **wrong** period (copy the cell above and set `period = 3.4`). What happens to the dip? Why?
2. 🟢 Fold on **7.0** days. Something different happens. What, and why is 7.0 special?
3. 🟡 Change the planet to 1.0 R⊕ (rebuild `flux`!). Is it visible after folding? What is the SNR?
4. 🔴 Write a loop over radii from 3.0 down to 0.5 R⊕ that prints the expected SNR for each. What's the
   smallest planet with SNR ≥ 7? **Write that number down**: in Week 4 you'll measure the real limit of your own
   pipeline and compare.

<details><summary>💡 Hints</summary>

1. On a wrong period the transits land at different phases each time, so they smear out.
2. 7.0 = 2 × 3.5. Every transit still lands somewhere: half at phase 0, half at ±3.5 d.
4. `for rp in np.arange(3.0, 0.4, -0.25):` and reuse `transit_depth` and `N`.
</details>

<details><summary>✅ Solution to 4</summary>

```python
for rp in np.arange(3.0, 0.4, -0.25):
    s = transit_depth(rp, 1.0) / (noise_ppm * 1e-6) * np.sqrt(N)
    print(f"{rp:4.2f} R_earth: SNR = {s:5.1f}", "✓" if s >= 7 else "")
```
SNR scales as $R_p^2$, so halving the radius quarters the SNR. Here SNR = 7 is reached at ≈ 2.2 R⊕: a 2.5 R⊕
planet is only just detectable in one sector of this noisy star. More sectors (more transits) or a quieter star push the limit down.
</details>

In [ ]:
# ✏️ YOUR CODE HERE: experiment freely

## 🧠 Concept check
**Q1.** A planet 2× bigger in radius makes a transit how many times deeper?
<details><summary>Answer</summary>4× (depth ∝ R²).</details>

**Q2.** You average 100 points that each have 500 ppm of noise. What is the noise of the average?
<details><summary>Answer</summary>500/√100 = 50 ppm.</details>

**Q3.** `a = np.array([3, 8, 1, 9])`. What is `a[a > 2]`? And `(a > 2).sum()`?
<details><summary>Answer</summary>`array([3, 8, 9])` and `3`.</details>

**Q4.** Why does a robust (median) average help when looking for transits?
<details><summary>Answer</summary>A few deviating points (outliers, or the transit itself) barely move the median, so a smoother built on medians follows the star but not the transit.</details>

## 📌 Key takeaways
- Transit depth $\delta = (R_p/R_\star)^2$: small planets, tiny dips (Earth/Sun ≈ 84 ppm).
- Noise on an average of $N$ points falls as $1/\sqrt{N}$, so SNR ≈ $(\delta/\sigma)\sqrt{N}$.
- NumPy arrays + boolean masks do 90 % of the work in this project.
- Read error messages from the bottom up. Run cells top to bottom.
- Test methods on fake data with a known answer.

## 📖 Glossary
**array**: a NumPy list of numbers you can do maths on · **mask**: an array of True/False used to select data ·
**ppm**: parts per million (×10⁻⁶) · **phase fold**: wrap time onto one period so transits stack ·
**SNR**: signal-to-noise ratio · **seed**: fixes "random" numbers so results are reproducible.

### ✅ Checkpoint (commit this notebook to your repo)
- [ ] Notebook runs top to bottom without errors (*Restart and run all*)
- [ ] All `check(...)` lines print ✅
- [ ] Your answers to Your Turn 5 in a markdown cell, in your own words
- [ ] One plot you are proud of, with labels and units

**➡️ Next:** Notebook 01, where you download a real light curve of a real planet from NASA's archive.